# Lab 06 — Support Vector Machine (SVM)
**Học máy ứng dụng · Nguyễn Hoàng Hải · 2474802010099**

Notebook bám theo tài liệu Lab06_SVM.pdf. Chạy lần lượt từ trên xuống. Bộ dữ liệu CSV được tìm ở thư mục hiện tại hoặc `data/`.

# Bài tập 06 — sáu bài trong tài liệu
Mỗi phần thực hiện đúng yêu cầu cuối PDF. Các kết quả có thể hơi khác về thời gian hoặc tie-break giữa phiên bản thư viện.

## Bài 1 — Tính tay khoảng cách tới ranh giới

In [ ]:
from sklearn.datasets import make_blobs
from sklearn.svm import SVC
X24,y24=make_blobs(n_samples=24,centers=[[2,2],[6,6]],cluster_std=.9,random_state=0); y24=1-y24
m24=SVC(kernel='linear',C=1000).fit(X24,y24); w,b=m24.coef_[0],m24.intercept_[0]
pts=np.array([[3.,3.],[5.,4.5]])
for p in pts:
    f=w[0]*p[0]+w[1]*p[1]+b
    print(f'{p}: f={f:+.4f}, khoảng cách={abs(f)/np.linalg.norm(w):.4f}')
print('Ba support vectors:')
for i in m24.support_:
    f=w[0]*X24[i,0]+w[1]*X24[i,1]+b
    print(i, X24[i], 'f=',round(f,4),'distance=',round(abs(f)/np.linalg.norm(w),4))
print('Nửa độ rộng lề:',1/np.linalg.norm(w))

**Giải thích:** Với support vector trên mép lề, |f(x)|=1. Do đó khoảng cách đến ranh giới là 1/||w||, đúng bằng một nửa độ rộng lề 2/||w||.

## Bài 2 — Thêm điểm và xem ranh giới

In [ ]:
def fit_in_ra_nhanh(X,y):
    m=SVC(kernel='linear',C=1000).fit(X,y)
    return m.coef_[0],m.intercept_[0],m
w0,b0,m0=fit_in_ra_nhanh(X24,y24)
for point,label in [([-1.,-1.],1),([4.5,4.],1)]:
    Xn=np.vstack([X24,point]); yn=np.append(y24,label)
    wnew,bnew,mnew=fit_in_ra_nhanh(Xn,yn)
    print(f'Thêm {point}, lớp {label}: w={wnew.round(4)}, b={bnew[0]:.4f}, support mới={len(mnew.support_)}')
    print('Khoảng cách giữa w mới và cũ:',np.linalg.norm(wnew-w0),'| Δb:',abs(bnew[0]-b0[0]))

**Nhận xét:** Điểm xa lề thường không đổi ranh giới; điểm nằm gần ranh giới có thể trở thành vectơ hỗ trợ và làm thay đổi nghiệm. So sánh hệ số và chỉ số support vectors để kết luận cho dữ liệu thực tế.

## Bài 3 — Dò C cho kernel đa thức

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split

def doc_csv(ten):
    ung_vien = [Path('data') / ten, Path(ten)]
    if ten == 'sinh_vien.csv':
        ung_vien += [Path('sinh_vien (1).csv'), Path('data/sinh_vien (1).csv')]
    for p in ung_vien:
        if p.exists():
            print(f'Đọc dữ liệu: {p}')
            return pd.read_csv(p)
    raise FileNotFoundError(f'Không tìm thấy {ten}. Đã tìm: {ung_vien}')

In [ ]:
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV
from sklearn.svm import SVC
df=doc_csv('hai_vang_trang.csv'); X=df[['x1','x2']]; y=df['nhan']
X_tr,X_te,y_tr,y_te=train_test_split(X,y,test_size=.25,random_state=17,stratify=y)
poly_results=[]
for degree in [2,3,4]:
    pipe=make_pipeline(StandardScaler(),SVC(kernel='poly',degree=degree))
    gs=GridSearchCV(pipe,{'svc__C':[.1,1,10,100]},cv=5).fit(X_tr,y_tr)
    poly_results.append({'degree':degree,'best_C':gs.best_params_['svc__C'],'cv':gs.best_score_,'test':gs.score(X_te,y_te)})
pd.DataFrame(poly_results).round(4)
print('Mốc RBF mục 5: CV≈0.8711, test≈0.9200')

## Bài 4 — Dò lưới mịn quanh nghiệm

In [ ]:
fine=GridSearchCV(SVC(kernel='rbf'),{'C':[3,10,30],'gamma':[.3,1,3]},cv=5).fit(X_tr,y_tr)
print('Cặp tốt nhất:',fine.best_params_)
print(f'CV={fine.best_score_:.4f}, test={fine.score(X_te,y_te):.4f}')
fine_table=pd.DataFrame(fine.cv_results_['params']); fine_table['cv']=fine.cv_results_['mean_test_score']
display(fine_table.pivot(index='C',columns='gamma',values='cv').round(4))
print('So với lưới thưa: CV tham chiếu 0.8711, test 0.9200')

**Nhận xét:** Lưới mịn có thể tìm điểm CV tốt hơn, nhưng mức tăng nhỏ và không đảm bảo cải thiện test; chọn tham số bằng CV chỉ dùng tập huấn luyện.

## Bài 5 — Confusion matrix và ảnh đoán sai

In [ ]:
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.datasets import load_digits
Xdig,ydig=load_digits(return_X_y=True)
Xd_tr,Xd_te,yd_tr,yd_te=train_test_split(Xdig,ydig,test_size=.25,random_state=17,stratify=ydig)
dig_svm=SVC(kernel='rbf').fit(Xd_tr,yd_tr); dig_pred=dig_svm.predict(Xd_te)
cm=confusion_matrix(yd_te,dig_pred,labels=np.arange(10))
print('Hàng là nhãn thật, cột là nhãn dự đoán'); display(pd.DataFrame(cm,index=range(10),columns=range(10)))
ConfusionMatrixDisplay(cm,display_labels=range(10)).plot(cmap='Blues',values_format='d'); plt.title('Confusion matrix — SVM RBF')
wrong=np.flatnonzero(dig_pred!=yd_te)
out=Path('anh_doan_sai'); out.mkdir(exist_ok=True)
for i in wrong:
    fig,ax=plt.subplots(figsize=(2,2)); ax.imshow(Xd_te[i].reshape(8,8),cmap='gray_r'); ax.set_title(f'thật {yd_te[i]} → đoán {dig_pred[i]}'); ax.axis('off')
    fig.savefig(out/f'anh_{i:03d}_that_{yd_te[i]}_doan_{dig_pred[i]}.png',bbox_inches='tight'); plt.close(fig)
print(f'Đã lưu {len(wrong)} ảnh ở {out.resolve()}')
print('Các cặp nhầm (thật → đoán, số lần):')
for a,b in zip(*np.where(cm-np.diag(np.diag(cm)))): print(f'{a} → {b}: {cm[a,b]}')
print('Từng ảnh nhầm được lưu để xem; nhiều lỗi đến từ nét viết tay mờ hoặc hình dạng chữ số gần nhau.')

## Bài 6 — Thời gian học trên dữ liệu lớn

In [ ]:
from sklearn.datasets import make_classification
from sklearn.svm import LinearSVC
from time import perf_counter
rows=[]
for n in [2000,5000,10000,20000]:
    Xbig,ybig=make_classification(n_samples=n,n_features=20,random_state=0)
    for name,model in [('SVC',SVC()),('LinearSVC',LinearSVC(random_state=0,max_iter=10000))]:
        t0=perf_counter(); model.fit(Xbig,ybig); elapsed=perf_counter()-t0
        rows.append({'n':n,'model':name,'fit_seconds':elapsed})
timings=pd.DataFrame(rows)
pivot=timings.pivot(index='n',columns='model',values='fit_seconds')
for col in pivot:
    pivot[col+'_factor_vs_previous']=pivot[col]/pivot[col].shift(1)
display(pivot.round(4))
print('Thời gian phụ thuộc máy và thư viện; so sánh hệ số tăng giữa các cỡ liền kề.')